In [ ]:
import os

model_dir = '/content/model_v1'

print("=" * 55)
print("DIAGNOSIS: /content/model_v1")
print("=" * 55)

if not os.path.exists(model_dir):
    print("❌ /content/model_v1 does not exist at all")
    print("   → You need to run training (CELL 5)")
else:
    print(f"✅ Folder exists")
    all_files = []
    for root, dirs, files in os.walk(model_dir):
        for f in files:
            fp = os.path.join(root, f)
            sz = os.path.getsize(fp)
            all_files.append((fp, sz))
            print(f"   {fp}  ({sz/1e6:.2f} MB)")

    if not all_files:
        print("   ⚠️  EMPTY FOLDER — model was never saved properly")
        print("   → Run CELL 5 to retrain and save correctly")
    else:
        total = sum(s for _, s in all_files)
        print(f"\n   Total size: {total/1e6:.1f} MB across {len(all_files)} files")

print()
for a in ['/content/labels.npy', '/content/training_config.json']:
    ex = os.path.exists(a)
    sz = os.path.getsize(a) if ex else 0
    print(f"  {'✅' if ex and sz > 0 else '❌'} {a}  ({sz} bytes)")



get_ipython().system('pip install -q scikit-learn matplotlib psutil')
print("✅ Done")

def find_dataset(base='/content'):
    for entry in sorted(os.listdir(base)):
        full = os.path.join(base, entry)
        if not os.path.isdir(full) or entry.startswith('.'): continue
        subs = [d for d in os.listdir(full)
                if os.path.isdir(os.path.join(full, d))]
        if len(subs) >= 5:
            if any(any(f.lower().endswith(('.jpg','.jpeg','.png'))
                       for f in os.listdir(os.path.join(full, s)))
                   for s in subs[:3]):
                return full
    return None

if os.path.exists(DATASET_PATH):
    dataset_path = DATASET_PATH
    print(f"✅ Found: {dataset_path}")
else:
    print(f"⚠️  '{DATASET_PATH}' not found — scanning /content ...")
    dataset_path = find_dataset()
    if dataset_path:
        print(f"✅ Auto-detected: {dataset_path}")
    else:
        raise RuntimeError("❌ No dataset found! Check the 📁 file browser and set DATASET_PATH.")

classes = sorted([d for d in os.listdir(dataset_path)
                  if os.path.isdir(os.path.join(dataset_path, d))])
total   = sum(
    len([f for f in os.listdir(os.path.join(dataset_path, c))
         if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
    for c in classes)

print(f"   Classes : {len(classes)} → {classes}")
print(f"   Images  : {total} | ~{total//max(len(classes),1)} per class")

import tensorflow as tf, sys
print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {tf.keras.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")

if not tf.config.list_physical_devices('GPU'):
    print("⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU")
else:
    for gpu in tf.config.list_physical_devices('GPU'):
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy("float32")
    print("✅ GPU ready | float32 precision set")


import json, time
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# ── Config ────────────────────────────────────────────────────────
BACKBONE    = "mobilenetv2"
BATCH_SIZE  = 16          # 16 is safe for Colab free RAM
EPOCHS_P1   = 10          # Phase 1: frozen backbone
EPOCHS_P2   = 5           # Phase 2: fine-tune top layers
DENSE_UNITS = 256
DROPOUT     = 0.4
IMG_SIZE    = (224, 224)
MODEL_DIR   = "/content/model_v1"
MODEL_PATH  = f"{MODEL_DIR}/model.keras"   # Keras 3 native format
CKPT_PATH   = f"{MODEL_DIR}/best.keras"
LOG_PATH    = f"{MODEL_DIR}/training_log.csv"
LABELS_PATH = "/content/labels.npy"
CONFIG_PATH = "/content/training_config.json"
SEED        = 42

os.makedirs(MODEL_DIR, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── Step 1: Discover labels ───────────────────────────────────────
print("=" * 55)
print("STEP 1 — Discovering labels")
print("=" * 55)

class_names = sorted([
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
])
label_map   = {name: i for i, name in enumerate(class_names)}
NUM_CLASSES = len(class_names)
print(f"Classes ({NUM_CLASSES}): {class_names}")

# ── Step 2: Collect image paths ───────────────────────────────────
print("\n" + "=" * 55)
print("STEP 2 — Collecting image paths")
print("=" * 55)

all_paths, all_labels = [], []
VALID_EXT = ('.jpg', '.jpeg', '.png', '.bmp')

for cls, idx in label_map.items():
    cls_dir = os.path.join(dataset_path, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(VALID_EXT):
            all_paths.append(os.path.join(cls_dir, fname))
            all_labels.append(idx)

print(f"Total images: {len(all_paths)}")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.2, random_state=SEED, stratify=all_labels
)
print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

# ── Step 3: Build tf.data pipelines ──────────────────────────────
print("\n" + "=" * 55)
print("STEP 3 — Building tf.data pipelines (no cache = OOM safe)")
print("=" * 55)

AUTOTUNE = tf.data.AUTOTUNE

def parse_image(path, label):
    img = tf.image.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.cast(tf.image.resize(img, IMG_SIZE), tf.float32)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, tf.one_hot(label, NUM_CLASSES)

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    return img, label

train_ds = (
    tf.data.Dataset.zip((
        tf.data.Dataset.from_tensor_slices(train_paths),
        tf.data.Dataset.from_tensor_slices(train_labels)
    ))
    .shuffle(min(len(train_paths), 2000), reshuffle_each_iteration=True)
    .map(parse_image, num_parallel_calls=AUTOTUNE)
    .map(augment,     num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(AUTOTUNE)
    # NOTE: .cache() intentionally omitted — prevents RAM OOM (return code -9)
)

val_ds = (
    tf.data.Dataset.zip((
        tf.data.Dataset.from_tensor_slices(val_paths),
        tf.data.Dataset.from_tensor_slices(val_labels)
    ))
    .map(parse_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("✅ Pipelines ready (no cache — RAM safe)")

# ── Step 4: Build model ───────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 4 — Building MobileNetV2 model")
print("=" * 55)

base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    alpha=1.0
)
base.trainable = False   # Phase 1: freeze backbone

inp = tf.keras.Input((224, 224, 3), name="image_input")
x   = base(inp, training=False)
x   = tf.keras.layers.GlobalAveragePooling2D()(x)
x   = tf.keras.layers.Dense(
          DENSE_UNITS, activation="relu",
          kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x   = tf.keras.layers.BatchNormalization()(x)
x   = tf.keras.layers.Dropout(DROPOUT)(x)
x   = tf.keras.layers.Dense(
          NUM_CLASSES, activation="softmax",
          kernel_regularizer=tf.keras.regularizers.l2(1e-4),
          name="output_softmax")(x)
# Cast to float32 — prevents TopKCategoricalAccuracy TypeError on GPU
out = tf.keras.layers.Activation(
          "linear", dtype="float32", name="output_float32")(x)

model = tf.keras.Model(inp, out, name="mobilenetv2_signlens")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy",
             tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")]
)

total     = model.count_params()
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  (head only)")
print(f"Frozen params    : {total - trainable:,}  (MobileNetV2 backbone)")

# ── Step 5: Phase 1 training ──────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 5 — Phase 1: Training head only (backbone frozen)")
print("=" * 55)

callbacks_p1 = [
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_PATH, monitor="val_accuracy",
        save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.3,
        patience=4, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.CSVLogger(LOG_PATH, append=False),
]

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P1,
    callbacks=callbacks_p1,
    verbose=1
)

best_p1 = max(history_p1.history.get("val_accuracy", [0]))
print(f"\n✅ Phase 1 complete — best val_accuracy: {best_p1:.4f}")

# ── Step 6: Phase 2 fine-tuning ───────────────────────────────────
print("\n" + "=" * 55)
print("STEP 6 — Phase 2: Fine-tuning top backbone layers")
print("=" * 55)

base.trainable = True
for layer in base.layers[:100]:  # keep early layers frozen
    layer.trainable = False

unfrozen = sum(1 for l in base.layers if l.trainable)
print(f"Unfrozen {unfrozen}/{len(base.layers)} base layers")

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy",
             tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")]
)

callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_PATH, monitor="val_accuracy",
        save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.3,
        patience=3, min_lr=1e-8, verbose=1),
    tf.keras.callbacks.CSVLogger(LOG_PATH, append=True),
]

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P2,
    callbacks=callbacks_p2,
    verbose=1
)

best_p2 = max(history_p2.history.get("val_accuracy", [0]))
print(f"\n✅ Phase 2 complete — best val_accuracy: {best_p2:.4f}")

# ── Step 7: Save everything ───────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 7 — Saving artifacts")
print("=" * 55)

# 1. labels.npy  ← FIRST (most critical, smallest, fastest)
np.save(LABELS_PATH, np.array(class_names))
print(f"✅ labels.npy saved  ({NUM_CLASSES} classes)")

# 2. training_config.json  ← SECOND
config = {
    "backbone": BACKBONE,
    "num_classes": NUM_CLASSES,
    "class_names": class_names,
    "label_map": label_map,
    "input_shape": [224, 224, 3],
    "model_path": MODEL_PATH,
    "confidence_threshold": 0.60,
    "best_val_accuracy_phase1": float(best_p1),
    "best_val_accuracy_phase2": float(best_p2),
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=4)
print(f"✅ training_config.json saved")

# 3. model.keras  ← THIRD  (Keras 3 native .keras format)
print(f"   Saving model to {MODEL_PATH} ...")
model.save(MODEL_PATH)
print(f"✅ model.keras saved  ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)")

# 4. Verify checkpoint exists
if os.path.exists(CKPT_PATH):
    print(f"✅ best.keras exists  ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)")

# 5. Training curves  ← LAST (non-critical)
try:
    import matplotlib.pyplot as plt
    all_acc  = history_p1.history["accuracy"]     + history_p2.history["accuracy"]
    val_acc  = history_p1.history["val_accuracy"] + history_p2.history["val_accuracy"]
    all_loss = history_p1.history["loss"]         + history_p2.history["loss"]
    val_loss = history_p1.history["val_loss"]     + history_p2.history["val_loss"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(all_acc,  label="Train Acc"); ax1.plot(val_acc,  label="Val Acc")
    ax1.axvline(x=len(history_p1.history["accuracy"])-1,
                color='gray', linestyle='--', label="Phase 2 start")
    ax1.set_title("Accuracy"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(all_loss, label="Train Loss"); ax2.plot(val_loss, label="Val Loss")
    ax2.set_title("Loss"); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("/content/training_curves.png", dpi=150)
    plt.show()
    print("✅ training_curves.png saved")
except Exception as e:
    print(f"⚠️  Plot failed (non-critical): {e}")

print("\n" + "=" * 55)
print("ALL ARTIFACTS SAVED — Summary:")
print("=" * 55)
for path in [MODEL_PATH, CKPT_PATH, LABELS_PATH, CONFIG_PATH]:
    ex = os.path.exists(path)
    sz = f"{os.path.getsize(path)/1e6:.2f} MB" if ex else "missing"
    print(f"  {'✅' if ex else '❌'} {path}  ({sz})")

import numpy as np
from sklearn.metrics import classification_report

print("Running classification report on validation set...")

y_true, y_pred = [], []
for imgs, lbls in val_ds:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(lbls.numpy(), axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

import random, time
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def predict_image(img_path, threshold=0.60):
    raw  = tf.io.read_file(img_path)
    img  = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img  = tf.cast(tf.image.resize(img, (224, 224)), tf.float32)
    img  = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    batch = tf.expand_dims(img, 0).numpy()
    t0   = time.perf_counter()
    probs = model(batch, training=False).numpy()[0]
    ms   = (time.perf_counter() - t0) * 1000
    idx  = int(np.argmax(probs))
    conf = float(probs[idx])
    return {
        "label": class_names[idx] if conf >= threshold else "UNCERTAIN",
        "raw": class_names[idx], "confidence": conf, "ms": round(ms, 1)
    }

test_classes = random.sample(class_names, min(6, len(class_names)))
fig, axes    = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle("Inference Test — Model Predictions", fontsize=14)

for ax, cls in zip(axes.flatten(), test_classes):
    cls_path = os.path.join(dataset_path, cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if not imgs: ax.axis('off'); continue
    img_path = os.path.join(cls_path, random.choice(imgs))
    r        = predict_image(img_path)
    correct  = r['raw'] == cls
    ax.imshow(mpimg.imread(img_path))
    ax.set_title(
        f"True: {cls}   Pred: {r['raw']} {'✓' if correct else '✗'}\n"
        f"Conf: {r['confidence']*100:.1f}%  |  {r['ms']}ms",
        color='green' if correct else 'red', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/inference_test.png', dpi=100)
plt.show()
print("✅ Inference test done!")


MODEL_PATH  = "/content/model_v1/model.keras"
LABELS_PATH = "/content/labels.npy"

print("Loading saved model from disk to verify...")

loaded_model = tf.keras.models.load_model(MODEL_PATH)
loaded_model.trainable = False

# Warm-up
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
out   = loaded_model.predict(dummy, verbose=0)

class_names_loaded = np.load(LABELS_PATH, allow_pickle=True).tolist()
pred_class = class_names_loaded[int(np.argmax(out[0]))]

print(f"✅ Model loaded successfully!")
print(f"   Output shape : {out.shape}")
print(f"   Test pred    : {pred_class}  (confidence: {float(np.max(out[0])):.4f})")
print(f"   Classes      : {class_names_loaded}")
print(f"\n✅ Model is valid and ready to download!")


import shutil, os
from google.colab import files

print("📦 Creating model_v1.zip ...")
shutil.make_archive('/content/model_v1_export', 'zip', '/content', 'model_v1')
sz = os.path.getsize('/content/model_v1_export.zip') / 1e6
print(f"✅ model_v1_export.zip ({sz:.1f} MB)")

print("\n📥 Starting downloads...")
files.download('/content/model_v1_export.zip')
files.download('/content/labels.npy')
files.download('/content/training_config.json')

for extra in ['/content/training_curves.png', '/content/inference_test.png']:
    if os.path.exists(extra):
        files.download(extra)
